# Feature Engineering - Route 502 Pilot

Bu notebook, `trip_extractor.py` tarafindan uretilen `route_502_segments.csv` dosyasini alir ve:

1. **Zamansal ozellikler** ekler (time_block, is_weekend)
2. **GTFS planlanmis sureler** ile eslestirir (scheduled_travel_time, deviation)
3. **Mekansal ozellikler** ekler (mesafe, durak orani)
4. **Hava durumu** verisi ile birlestirir
5. ML-ready `route_502_features.csv` uretir

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import csv
import math
import os
import sys

# Proje kokleri
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
COLLECTOR_DIR = os.path.join(PROJECT_ROOT, 'data_collector')
GTFS_DIR = os.path.join(PROJECT_ROOT, 'data', 'bus-eshot-gtfs')
SEGMENTS_CSV = os.path.join(COLLECTOR_DIR, 'collected_data', 'extracted_trips', 'route_502_segments.csv')
DB_PATH = os.path.join(COLLECTOR_DIR, 'collected_data', 'route_502_realtime.db')
OUTPUT_CSV = os.path.join(COLLECTOR_DIR, 'collected_data', 'extracted_trips', 'route_502_features.csv')

# config.py'yi import et
sys.path.insert(0, COLLECTOR_DIR)
from config import STOPS_DIR0, STOPS_DIR1

print(f'Proje kok: {PROJECT_ROOT}')
print(f'Segments CSV: {os.path.exists(SEGMENTS_CSV)}')
print(f'GTFS dir: {os.path.exists(GTFS_DIR)}')
print(f'DB: {os.path.exists(DB_PATH)}')

## 1. Segments CSV Yukle

In [ ]:
df = pd.read_csv(SEGMENTS_CSV)
print(f'Toplam segment: {len(df)}')
print(f'Kolonlar: {list(df.columns)}')
print(f'\nSure istatistikleri (dk):')
print(df['travel_minutes'].describe())
df.head()

## 2. Zamansal Ozellikler

In [ ]:
# is_weekend: 5=Cumartesi, 6=Pazar
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

# time_block: makale standardinda 4 dilim
def get_time_block(hour):
    if 6 <= hour < 10:
        return 'morning_peak'
    elif 10 <= hour < 17:
        return 'off_peak'
    elif 17 <= hour < 20:
        return 'evening_peak'
    else:
        return 'night'

df['time_block'] = df['hour'].apply(get_time_block)

print('Zaman bloklari dagilimi:')
print(df['time_block'].value_counts())
print(f'\nis_weekend dagilimi: {df["is_weekend"].value_counts().to_dict()}')

## 3. GTFS Planlanmis Sureler (Ozgun Katki)

Makalede kullanilmayan ozellik: GTFS'teki planlanmis duraklar arasi sureyi feature olarak ekliyoruz.

In [ ]:
# Route 502 trip'lerini bul
trips_gtfs = pd.read_csv(os.path.join(GTFS_DIR, 'trips.txt'))
route502_trip_ids = set(trips_gtfs[trips_gtfs['route_id'] == 502]['trip_id'].values)
print(f'Route 502 GTFS trip sayisi: {len(route502_trip_ids)}')

# stop_times'tan Route 502 verilerini al
stop_times = pd.read_csv(os.path.join(GTFS_DIR, 'stop_times.txt'))
st502 = stop_times[stop_times['trip_id'].isin(route502_trip_ids)].copy()
print(f'Route 502 stop_times kayit: {len(st502)}')

In [ ]:
def time_str_to_seconds(t):
    """HH:MM:SS formatini saniyeye cevir."""
    parts = t.strip().split(':')
    return int(parts[0]) * 3600 + int(parts[1]) * 60 + int(parts[2])


# Her trip icin ardisik duraklar arasi planlanmis sureyi hesapla
st502['arr_sec'] = st502['arrival_time'].apply(time_str_to_seconds)
st502['dep_sec'] = st502['departure_time'].apply(time_str_to_seconds)
st502 = st502.sort_values(['trip_id', 'stop_sequence'])

# Her (from_seq, to_seq) cifti icin ortalama planlanmis sureyi hesapla
scheduled_records = []
for trip_id, group in st502.groupby('trip_id'):
    group = group.sort_values('stop_sequence')
    seqs = group['stop_sequence'].values
    deps = group['dep_sec'].values
    arrs = group['arr_sec'].values
    
    for i in range(1, len(seqs)):
        # Planlanmis sure: onceki duragin departure -> su anki duragin arrival
        sched_seconds = arrs[i] - deps[i - 1]
        if sched_seconds < 0:
            continue  # Gece yarisindan gecen zamanlar
        scheduled_records.append({
            'from_seq': int(seqs[i - 1]),
            'to_seq': int(seqs[i]),
            'scheduled_seconds': sched_seconds,
        })

sched_df = pd.DataFrame(scheduled_records)

# (from_seq, to_seq) bazinda ortalama
sched_avg = sched_df.groupby(['from_seq', 'to_seq'])['scheduled_seconds'].mean().reset_index()
sched_avg.rename(columns={'scheduled_seconds': 'scheduled_travel_seconds'}, inplace=True)
sched_avg['scheduled_travel_minutes'] = (sched_avg['scheduled_travel_seconds'] / 60.0).round(2)

print(f'Benzersiz segment (from_seq, to_seq) cifti: {len(sched_avg)}')
print(f'\nPlanlanmis sure istatistikleri (dk):')
print(sched_avg['scheduled_travel_minutes'].describe())
sched_avg.head(10)

In [ ]:
# Segments tablosundaki seq'ler azaliyor (8->7, 7->6, ...)
# GTFS'teki seq'ler artiyor (1->2, 2->3, ...)
# Eslestirmek icin: segments'teki (from=8, to=7) GTFS'teki (from=7, to=8) ile ayni segment
#
# Yani: segments.from_stop_seq -> sched_avg.to_seq
#        segments.to_stop_seq   -> sched_avg.from_seq

df = df.merge(
    sched_avg[['from_seq', 'to_seq', 'scheduled_travel_seconds', 'scheduled_travel_minutes']],
    left_on=['to_stop_seq', 'from_stop_seq'],   # ters cevir: segments to=GTFS from
    right_on=['from_seq', 'to_seq'],
    how='left'
).drop(columns=['from_seq', 'to_seq'])

# Deviation: gercek - planlanmis (pozitif = gecikme, negatif = erkenci)
df['deviation_seconds'] = df['travel_seconds'] - df['scheduled_travel_seconds']
df['deviation_minutes'] = (df['deviation_seconds'] / 60.0).round(2)

matched = df['scheduled_travel_minutes'].notna().sum()
print(f'GTFS eslesmesi: {matched}/{len(df)} ({matched/len(df)*100:.0f}%)')
print(f'\nDeviation istatistikleri (dk):')
print(df['deviation_minutes'].describe())

## 4. Mekansal Ozellikler

In [ ]:
def haversine_m(lat1, lon1, lat2, lon2):
    """Iki koordinat arasi mesafe (metre)."""
    R = 6371000
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlam = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlam / 2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))


# Durak listelerinden seq -> koordinat sozlugu
stop_coords = {}
for s in STOPS_DIR0:
    stop_coords[(0, s['seq'])] = (s['lat'], s['lon'], s['name'])
for s in STOPS_DIR1:
    stop_coords[(1, s['seq'])] = (s['lat'], s['lon'], s['name'])

# Her segment icin from ve to durak arasi mesafe
distances = []
for _, row in df.iterrows():
    yon = int(row['yon'])
    from_key = (yon, int(row['from_stop_seq']))
    to_key = (yon, int(row['to_stop_seq']))
    if from_key in stop_coords and to_key in stop_coords:
        lat1, lon1, _ = stop_coords[from_key]
        lat2, lon2, _ = stop_coords[to_key]
        distances.append(round(haversine_m(lat1, lon1, lat2, lon2), 1))
    else:
        distances.append(np.nan)

df['distance_m'] = distances

# Durak orani: hattın yuzde kaci gecildi (0-1)
# Yon 0: STOPS_DIR0 -> 32 durak, Yon 1: STOPS_DIR1 -> 28 durak
max_seq = {0: max(s['seq'] for s in STOPS_DIR0),
           1: max(s['seq'] for s in STOPS_DIR1)}

# seq azaliyor, yani from_stop_seq buyuk = hattın basinda
# normallestirilmis konum: 1 - (from_seq / max_seq) -> 0'a yakin = baslangic, 1'e yakin = bitis
df['stop_progress'] = df.apply(
    lambda r: round(1.0 - int(r['from_stop_seq']) / max_seq[int(r['yon'])], 3), axis=1
)

print(f'Mesafe istatistikleri (metre):')
print(df['distance_m'].describe())
print(f'\nstop_progress: min={df["stop_progress"].min()}, max={df["stop_progress"].max()}')

## 5. Hava Durumu Entegrasyonu

collector.py saatte bir `weather_readings` tablosuna hava durumu kaydediyor.  
Her segmentin timestamp'ine en yakin saatlik kayit ile eslestiriyoruz.

In [ ]:
conn = sqlite3.connect(DB_PATH)
weather_df = pd.read_sql_query('SELECT * FROM weather_readings ORDER BY timestamp', conn)
conn.close()

print(f'Hava durumu kayit sayisi: {len(weather_df)}')
if len(weather_df) > 0:
    print(weather_df.head())

In [ ]:
# Segmentin tarih+saat bilgisinden full timestamp olustur
df['segment_datetime'] = pd.to_datetime(df['date'] + ' ' + df['trip_start_time'])

if len(weather_df) > 0:
    weather_df['timestamp'] = pd.to_datetime(weather_df['timestamp'])
    
    # Her segment icin en yakin hava kaydini bul (merge_asof)
    df_sorted = df.sort_values('segment_datetime')
    weather_sorted = weather_df.sort_values('timestamp')
    
    merged = pd.merge_asof(
        df_sorted,
        weather_sorted[['timestamp', 'temperature', 'humidity', 'precipitation',
                        'wind_speed', 'visibility', 'weather_category', 'source']],
        left_on='segment_datetime',
        right_on='timestamp',
        direction='nearest',
        tolerance=pd.Timedelta('2h')  # 2 saatten uzak hava kaydini esleme
    )
    
    # is_rainy ikili ozellik
    merged['is_rainy'] = (merged['weather_category'] == 'rainy').astype(int)
    
    weather_matched = merged['temperature'].notna().sum()
    print(f'Hava durumu eslesmesi: {weather_matched}/{len(merged)}')
    print(f'Hava kaynak: {merged["source"].value_counts().to_dict()}')
    
    df = merged.drop(columns=['timestamp'], errors='ignore')
else:
    # Hava verisi henuz yok, bos kolonlar ekle
    for col in ['temperature', 'humidity', 'precipitation', 'wind_speed',
                'visibility', 'weather_category', 'is_rainy']:
        df[col] = np.nan
    print('Hava durumu verisi yok - kolonlar NaN olarak eklendi')

print(f'\nDataset boyutu: {df.shape}')

## 6. Final Dataset

In [ ]:
# Cikarilacak final kolonlar
feature_cols = [
    # Kimlik
    'date', 'bus_id', 'yon', 'trip_start_time',
    # Zamansal
    'hour', 'day_of_week', 'is_weekend', 'time_block',
    # Segment
    'from_stop_seq', 'to_stop_seq', 'from_stop_name', 'to_stop_name',
    # Mekansal
    'distance_m', 'stop_progress',
    # GTFS (ozgun katki)
    'scheduled_travel_seconds', 'scheduled_travel_minutes',
    # Hava durumu
    'temperature', 'humidity', 'precipitation', 'wind_speed',
    'visibility', 'weather_category', 'is_rainy',
    # Sapma
    'deviation_seconds', 'deviation_minutes',
    # Hedef degisken
    'travel_seconds', 'travel_minutes',
]

# Var olan kolonlari sec (hava verisi yoksa NaN olarak gelir)
available_cols = [c for c in feature_cols if c in df.columns]
df_final = df[available_cols].copy()

print(f'Final dataset: {df_final.shape[0]} satir, {df_final.shape[1]} kolon')
print(f'\nKolonlar:')
for c in df_final.columns:
    non_null = df_final[c].notna().sum()
    print(f'  {c}: {non_null}/{len(df_final)} non-null ({df_final[c].dtype})')

In [ ]:
# Ozet istatistikler
print('=== Sayisal Ozellikler ===')
df_final[['travel_minutes', 'scheduled_travel_minutes', 'deviation_minutes',
          'distance_m', 'stop_progress']].describe().round(2)

In [ ]:
# CSV'ye kaydet
df_final.to_csv(OUTPUT_CSV, index=False)
print(f'Kaydedildi: {OUTPUT_CSV}')
print(f'Boyut: {os.path.getsize(OUTPUT_CSV) / 1024:.1f} KB')
print(f'{len(df_final)} satir, {len(df_final.columns)} kolon')

## 7. Kesfedici Gorsellestirmeler

In [ ]:
try:
    import matplotlib.pyplot as plt
    HAS_PLT = True
except ImportError:
    HAS_PLT = False
    print('matplotlib bulunamadi, gorsellestirme atlanacak')

In [ ]:
if HAS_PLT and len(df_final) > 5:
    fig, axes = plt.subplots(2, 2, figsize=(12, 8))
    
    # 1. Gercek suresi dagilimi
    axes[0, 0].hist(df_final['travel_minutes'].dropna(), bins=20, edgecolor='black', alpha=0.7)
    axes[0, 0].set_xlabel('Seyahat suresi (dk)')
    axes[0, 0].set_ylabel('Frekans')
    axes[0, 0].set_title('Gercek seyahat suresi dagilimi')
    
    # 2. Gercek vs Planlanmis
    mask = df_final['scheduled_travel_minutes'].notna()
    if mask.sum() > 1:
        axes[0, 1].scatter(
            df_final.loc[mask, 'scheduled_travel_minutes'],
            df_final.loc[mask, 'travel_minutes'],
            alpha=0.5, s=15
        )
        # 45 derece cizgi
        lim = max(df_final.loc[mask, 'scheduled_travel_minutes'].max(),
                  df_final.loc[mask, 'travel_minutes'].max()) + 0.5
        axes[0, 1].plot([0, lim], [0, lim], 'r--', alpha=0.5)
        axes[0, 1].set_xlabel('Planlanmis (dk)')
        axes[0, 1].set_ylabel('Gercek (dk)')
        axes[0, 1].set_title('Planlanmis vs Gercek')
    
    # 3. Sapma dagilimi
    dev = df_final['deviation_minutes'].dropna()
    if len(dev) > 1:
        axes[1, 0].hist(dev, bins=20, edgecolor='black', alpha=0.7, color='orange')
        axes[1, 0].axvline(0, color='red', linestyle='--', alpha=0.7)
        axes[1, 0].set_xlabel('Sapma (dk)')
        axes[1, 0].set_ylabel('Frekans')
        axes[1, 0].set_title(f'Gecikme dagilimi (ort: {dev.mean():.2f} dk)')
    
    # 4. Durak bazinda ortalama sure
    if len(df_final) > 3:
        by_seq = df_final.groupby('from_stop_seq')['travel_minutes'].mean().sort_index()
        axes[1, 1].bar(by_seq.index.astype(str), by_seq.values, alpha=0.7, color='green')
        axes[1, 1].set_xlabel('From stop seq')
        axes[1, 1].set_ylabel('Ortalama sure (dk)')
        axes[1, 1].set_title('Durak bazinda ortalama seyahat suresi')
        axes[1, 1].tick_params(axis='x', rotation=45, labelsize=7)
    
    plt.tight_layout()
    plt.savefig(os.path.join(PROJECT_ROOT, 'results', 'figures', 'feature_engineering_eda.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    print('Gorsel kaydedildi: results/figures/feature_engineering_eda.png')
else:
    print('Yeterli veri yok veya matplotlib eksik, gorsellestirme atlandi')

## Sonuc

Uretilen `route_502_features.csv` icerigi:

| Grup | Ozellikler |
|------|------------|
| Hedef degisken | `travel_minutes`, `travel_seconds` |
| Zamansal | `hour`, `day_of_week`, `is_weekend`, `time_block` |
| Mekansal | `distance_m`, `stop_progress`, `from_stop_seq`, `to_stop_seq` |
| GTFS (ozgun katki) | `scheduled_travel_minutes`, `deviation_minutes` |
| Hava durumu | `temperature`, `humidity`, `precipitation`, `wind_speed`, `is_rainy` |

Sonraki adim: `baseline_models.ipynb` ile model egitimi.